# Working with JSON Bankruptcy Records

# Introduction

Real datasets almost never arrive as a clean spreadsheet. The bankruptcy
data for this project ships the way production data usually does:
**compressed, semi-structured JSON** — large, mildly messy, and optimized
for storage and transport rather than for analysis. Before we can model
anything, we have to turn those raw files into a tidy table we can trust.

That turning-raw-into-tidy step is the whole job of this notebook, and we
will package it into a single reusable function, `wrangle()`, that the
next three lessons import and reuse.

By the end of this notebook you will be able to:

-   Explain what JSON is (and how it differs from Python objects).
-   Load both `.json` and `.json.gz` files safely.
-   Inspect semi-structured data to understand its shape before using
    pandas.
-   Convert JSON records into a tidy DataFrame.
-   Clean and standardize features into a modeling-ready table.
-   Work with datasets **with and without** the target column.
-   Identify the target column and quantify class imbalance (when labels
    exist).

> 📌 **Tip — Data dictionary**
>
> If you want the meaning of each `feat_*` column (and the dataset
> structures), check out the **Data Dictionary lesson**.

# 1. Conceptual Foundation

Before we write a single line of loading code, we need a shared mental
model of *what* we are loading and *what* we want at the end. This
section builds that model: the project's definition of "done", the JSON
format itself, the gzip wrapper, the `with` statement, why pandas
sometimes refuses to read JSON directly, and what a "tidy" table means
here.

## Project context: what we're trying to build

The bankruptcy datasets are realistic: large, mildly messy, and stored
in formats optimized for storage and transport rather than analysis. Our
goal is to turn the raw files into DataFrames that are easy to model
later.

> ✅ **Definition of "ready for the next notebook"**
>
> -   One row per company (observation)
> -   One column per feature
> -   Features are numeric (non-numeric entries become missing values)
> -   If the file contains labels, there is a target column named
>     `bankrupt`

This notebook focuses on building a reliable **wrangle** step: a
repeatable transformation from raw files to a clean table.

➡️ First, we need to understand the format the data actually arrives in —
JSON.

## JSON: what it is, what it isn't, and how to debug it

JSON (JavaScript Object Notation) is a **text format** for structured
data. It looks a lot like Python — curly braces, square brackets, quotes
— but JSON is **stricter**, and that strictness is exactly what trips
people up.

> 🧠 **The key idea**
>
> JSON is *text on disk*. Python objects are *structures in memory*.
> Reading JSON means **parsing that text into Python objects**; writing
> JSON means **serializing objects back into text**.

### JSON ↔ Python mapping

Every JSON construct maps onto a familiar Python type. Read this table
the way you would read a translation dictionary:

| JSON | Python | Note |
|---|---|---|
| object `{…}` | `dict` | keys must be double-quoted strings |
| array `[…]` | `list` | ordered, can mix types |
| string `"…"` | `str` | **double quotes only** |
| number | `int` / `float` | no quotes |
| `true` / `false` | `True` / `False` | JSON is lowercase |
| `null` | `None` | not `None`, not `nil` |

> ⚠️ **The three rules that break most JSON**
>
> -   **Double quotes only** — single quotes are invalid.
> -   **No trailing commas** — the comma after the last item is illegal.
> -   **`true` / `false` / `null`** — not Python's `True` / `False` /
>     `None`.

➡️ Let's see a valid record first, then deliberately break it.

### Valid JSON example

The snippet below obeys every rule: double-quoted keys and strings,
lowercase `null`, no trailing commas. When we hand it to `json.loads`,
Python hands us back a `dict` we can index into.

**Code 5.1.1.1**:

In [ ]:
import json

good_json = """
{
  "company_id": 123,
  "country": "PL",
  "features": {
    "assets": 45000.0,
    "net_profit": -1200.5
  },
  "bankrupt": 1,
  "tags": ["manufacturing", "export"],
  "note": null
}
"""

obj = json.loads(good_json)
type(obj), obj["features"]["assets"]

### Malformed JSON example (on purpose)

This snippet breaks common JSON rules:

-   uses single quotes `'`
-   has a trailing comma
-   uses `True` instead of `true`

> 🔍 **What to look for** in the output below: `json.loads` does not just
> say "bad" — it reports the **line and column** where parsing failed.
> That location is your first clue when debugging real files.

**Code 5.1.1.2**:

In [ ]:
bad_json = """
{
'company_id': 123,
"features": {"assets": 45000.0, "net_profit": -1200.5,},
"bankrupt": True
}
"""

try:
    json.loads(bad_json)
except json.JSONDecodeError as err:
    print("JSONDecodeError:", err.msg)
    print("Line:", err.lineno, "Column:", err.colno)

> 🎥 **Walkthrough video**
>
> The short video below walks through the JSON wrangling workflow for
> this lesson. Watch it, then continue with the hands-on sections.

In [ ]:
from IPython.display import VimeoVideo

VimeoVideo("1170264290", h="3298dbabb7", width=700, height=450)

### Debugging malformed JSON (online linter workflow)

When you see `JSONDecodeError`, resist the urge to stare at the whole
file. A focused workflow is faster:

1.  Copy a **small snippet** around the suspicious region (not the whole
    file).

2.  Search online for "JSON linter" / "JSON validator".

3.  Paste the snippet and read the error message.

4.  Compare the highlighted location to the JSON rules:

    -   double quotes only
    -   no trailing commas
    -   `true/false/null`

> 📌 **Reminder:** the `JSONDecodeError` line/column you just saw points
> you straight at the snippet to paste into the linter.

## gzip: why `.json.gz` exists and how to read it

Large JSON files compress extremely well (lots of repeated keys and
whitespace), so datasets are commonly distributed as `.json.gz`.

> 🧠 **Key idea:** gzip is a **compression wrapper**, not a different
> data format. Inside a `.json.gz` file there is still ordinary JSON
> text — it is just squeezed. Unzip it on the fly and parse as usual.

### Text mode vs binary mode

When you open the file you must choose a mode, and the choice matters:

| Mode | Code | Use when | Gives you |
|---|---|---|---|
| **Text** | `"rt"` | parsing JSON with `json.load` | decoded `str` |
| **Binary** | `"rb"` | manually handling raw bytes | raw `bytes` |

> ✅ We will use `"rt"` (read-text) everywhere, because `json.load`
> expects decoded text, not bytes.

➡️ Reading files safely also means closing them reliably — which is what
the `with` statement is for.

## Context managers (`with`) and why we use them everywhere

When reading files, you'll see this pattern:

``` python
with open(...) as f:
    ...
```

> 💡 **Why `with`?**
>
> The `with` statement guarantees the file is properly **closed** even if
> an error happens during parsing. This matters when:
>
> -   files are large
> -   you re-run notebook cells often
> -   you read compressed streams (`gzip.open`)

Without `with`, a parsing error mid-read could leave a file handle open;
across many notebook re-runs that leaks resources. The context manager
makes cleanup automatic.

## Why pandas sometimes fails on JSON

Pandas is great at reading *tabular* data. But JSON often has a **root
object** that contains metadata **plus** the actual records.

A common structure looks like:

``` json
{
  "schema": {...},
  "metadata": {...},
  "data": [ {...record1...}, {...record2...}, ... ]
}
```

> ❗️ **But wait — `pd.read_json` on the whole file can fail.** Pandas
> may not know which part of that object is "the rows", so it errors out
> instead of guessing.

> ✅ **The fix is conceptual, not magic:**
>
> -   first **locate the list of records** (often `payload["data"]`, but
>     not always)
> -   then **convert that list** into a DataFrame

That two-step "find the records, then build the table" idea is the
backbone of the helper functions we write below.

## What "tidy DataFrame" means for this project

We will use the word **tidy** with a precise meaning, so every helper
function has the same target in mind:

| Aspect | Tidy rule for this project |
|---|---|
| Rows | each row = **one company record** |
| Columns | each column = **one feature** (`feat_1`, `feat_2`, …) |
| Target | if present, a single column `bankrupt` (binary 0/1) |

> 🧠 Everything that follows — loading, extracting, normalizing,
> cleaning — exists to move a raw `.json.gz` file into exactly this
> shape.

➡️ Concepts in hand, we switch from *understanding* to *doing*.

# Applied Exercises

## 2. Setup

Note: Import the required libraries and classes. It is highly
recommended to place all imports in a single cell at the beginning of
the notebook.

> 📦 We import `gzip` and `json` (for reading compressed JSON), `Path`
> (for filesystem paths), `Any` (for type hints on loosely-typed JSON),
> and `pandas` (our table tool). The `pd.set_option` lines just widen the
> display so we can see all the financial columns.

**Code 5.1.2.1**:

In [ ]:
from __future__ import annotations

import gzip
import json
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

## 3. Main goal

Build a function:

``` python
df = wrangle(path)
```

that returns a clean DataFrame from a `.json.gz` file.

> 🎯 We'll build it in **small, verifiable steps**. Each helper does one
> job, each gets a Checkpoint that `assert`s it works, and at the end we
> compose them into `wrangle()`.

## 4. Locate the dataset files

### Problem

We need to find the project data files in a predictable way.

### Approach

List `data/*.json*` using **glob**.

The [`glob`](https://docs.python.org/3/library/glob.html) module lets
you work with groups of files by matching their paths against simple
patterns, much like what you would do in a Unix shell. It searches the
filesystem for filenames that match wildcards such as `*` (any sequence
of characters), `?` (a single character), and character ranges defined
with square brackets (for example, `[a-z]`). Unlike a shell, it does not
expand the tilde (`~`) to represent the home directory.

> 🔍 **Key building blocks** (links to the docs):
>
> -   [sorted](https://docs.python.org/3/howto/sorting.html#sorting-basics)
>     — deterministic order, so results are reproducible
> -   [Path](https://docs.python.org/3/library/pathlib.html#basic-use)
>     — object-oriented filesystem paths
> -   [Path.glob](https://docs.python.org/3/library/pathlib.html#pathlib.Path.glob)
>     — pattern matching on a directory

**Code 5.1.4.1**:

In [ ]:
data_dir = Path("data")
paths = sorted(data_dir.glob("*.json*"))
paths

### Checkpoint

> 🧪 This assert guarantees the folder really contains the **four**
> expected files before we go further — if it fails, the data isn't where
> we think it is, and every later step would fail mysteriously.

**Code 5.1.4.2**:

In [ ]:
assert len(paths) == 4, "Inside data folder you should have four .json/.json.gz files."

### Group the 4 expected files

In the **data** folder, you will find two files for Taiwan and two files
for Poland. Create two variables, `poland_paths` and `taiwan_paths`,
each of which contains two `Path` objects for the corresponding files.

You can use a [list
comprehension](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions)
with an `if` condition to select only the desired paths.

For example:

``` python
my_variable = [p for p in mypaths if "any name" in p.name.lower()]
```

**Code 5.1.4.3**:

In [ ]:
poland_paths = sorted([p for p in paths if "poland" in p.name.lower()])
taiwan_paths = sorted([p for p in paths if "taiwan" in p.name.lower()])

poland_paths, taiwan_paths

### Checkpoint

> 🧪 These asserts confirm each country has exactly **2** files and that
> every path actually belongs to the country we filed it under — catching
> a mis-typed filter before it corrupts later steps.

**Code 5.1.4.4**:

In [ ]:
assert len(poland_paths) == 2, "Expected 2 Poland files"
assert len(taiwan_paths) == 2, "Expected 2 Taiwan files"

assert all(["poland" in str(p).lower() for p in poland_paths]), "Expected 2 Poland files"
assert all(["taiwan" in str(p).lower() for p in taiwan_paths]), "Expected 2 Taiwan files"

## 5. Load JSON safely (`load_json`)

> 💡 **Why this function?** Some files are plain `.json`, some are
> compressed `.json.gz`. Rather than remember which is which everywhere,
> we write **one** loader that detects the `.gz` suffix and opens the
> file correctly either way — always in text mode (`"rt"`), always inside
> a `with` block.

### Problem

All datasets are stored as gzip-compressed JSON files (`.json.gz`).

### Approach

Write a function named `load_json` that accepts a `path` argument, reads
the corresponding file, and returns its contents as a dictionary.

-   Use
    [`gzip.open(..., "rt", encoding="utf-8")`](https://docs.python.org/3/library/gzip.html)
-   Parse using
    [`json.load(...)`](https://docs.python.org/3/library/json.html)

**Code Task 5.1.5.1**:

In [ ]:
def load_json(path: str | Path) -> Any:
    """Load a JSON file that may be gzip-compressed.

    Parameters
    ----------
    path
        Path to a ``.json`` or ``.json.gz`` file.

    Returns
    -------
    Any
        Parsed JSON (often a dict, sometimes a list).
    """
    p = ...  # use Path

    if p.suffix == ".gz":
        with gzip.open(...) as f:
            return json.load(...)

    with p.open(...) as f:
        return json.load(...)

### Checkpoint: inspect one file

> 🔍 **What to look for:** the **root type** (is it a `dict`?) and its
> **top-level keys**. Those keys tell us *where the records live* — the
> exact question the next function answers.

**Code 5.1.5.2**:

In [ ]:
payload0 = load_json(paths[0])

print("Root type:", type(payload0))
if isinstance(payload0, dict):
    print("Top-level keys:", list(payload0.keys()))

assert payload0 is not None

> 📊 **Reading that output:** the payload is a `dict`, and among its
> top-level keys is the one holding the records (`data` for Poland,
> `observations` for Taiwan). We never see the records by reading the
> whole object — we have to reach into the right key, which motivates
> `extract_records` next.

## 6. Extract the record list from the payload (`extract_records`)

> 💡 **Why this function?** Poland and Taiwan store their records under
> **different keys**. Hard-coding one key would break the other dataset.
> This helper checks both, and fails loudly with a helpful message if
> neither is found.

### Problem

The record list key differs by dataset:

-   **Poland** uses `data`
-   **Taiwan** uses `observations`

### Approach (simple + explicit)

Create a function called `extract_records` that receives an argument
called payload and returns a list of dictionaries.

-   If `data` exists, use it.
-   Else if `observations` exists, use it.
-   Otherwise, raise a clear error and inspect keys.

> ⚠️ Notice the defensive checks in the solution: it confirms the payload
> is a `dict`, that the records container is a `list`, and that the first
> record is a `dict`. Failing early with a clear message beats a confusing
> crash three steps later.

**Code Task 5.1.6.1**:

In [ ]:
def extract_records(payload: Any) -> list[dict[str, Any]]:
    """
    Extract the list of record dictionaries from the dataset payload.

    Parameters
    ----------
    payload
        Parsed JSON payload (expected: dict with records under a known key).

    Returns
    -------
    list of dict
        The dataset records.

    Raises
    ------
    KeyError
        If neither ``data`` nor ``observations`` is present.
    TypeError
        If the record container exists but is not a list of dicts.
    """
    if not isinstance(payload, dict):
        raise TypeError("Expected the JSON payload to be a dict.")

    if "data" in payload:
        records = ...
    elif "observations" in payload:
        records = ...
    else:
        raise KeyError(
            "Could not find records. Expected key 'data' or 'observations'. "
            f"Found keys: {list(payload.keys())}"
        )

    if not isinstance(records, list):
        raise TypeError("Expected the records container to be a list.")

    if records and not isinstance(records[0], dict):
        raise TypeError("Expected records to be a list of dicts.")

    return records

### Checkpoint

> 🧪 Confirms we actually pulled out a **non-empty list of dicts** — the
> shape `records_to_frame` expects next.

**Code 5.1.6.2**:

In [ ]:
records0 = extract_records(payload0)

assert isinstance(records0, list)
assert len(records0) > 0
assert isinstance(records0[0], dict)

list(records0[0].keys())[:10]

## 7. Convert records to a DataFrame (`records_to_frame`)

> 💡 **Why this function?** A list of dicts is *almost* a table. Pandas
> has two ways to make the leap, and the right one depends on whether any
> values are themselves nested.

### `from_records` vs `json_normalize`

| Function | Use when | What it does with nesting |
|---|---|---|
| `pd.DataFrame.from_records` | values are flat scalars | keeps nested dict/list as a raw object in the cell |
| `pd.json_normalize` | values contain nested dicts/lists | **flattens** nested keys into dotted columns |

> 🧠 The solution checks for nesting first, then picks the right tool — so
> it works whether the records are flat or nested.

### Problem

We now have a list of dictionaries, with one dictionary per company. We
need to convert this list into a DataFrame.

### Approach

Create a function called `records_to_frame` that receives an argument
called `records` and returns a DataFrame.

-   Use
    [`pd.DataFrame.from_records(records)`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.from_records.html)
-   If nested dict/list values exist, use
    [`pd.json_normalize(records)`](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)

**Code Task 5.1.7.1**:

In [ ]:
def records_to_frame(records: list[dict[str, Any]]) -> pd.DataFrame:
    """
    Convert a list of record dicts into a DataFrame.

    Parameters
    ----------
    records
        List of dictionaries, one per observation.

    Returns
    -------
    pandas.DataFrame
        DataFrame created from records.
    """
    has_nested = any(
        isinstance(v, (dict, list))
        for r in records
        for v in r.values()
    )
    if has_nested:
        return ...
    return ...

### Checkpoint

> 🧪 Confirms we now hold a real DataFrame with **at least one row and one
> column** — i.e., the records survived the conversion.

**Code 5.1.7.2**:

In [ ]:
df_raw0 = records_to_frame(records0)

assert isinstance(df_raw0, pd.DataFrame)
assert df_raw0.shape[0] > 0
assert df_raw0.shape[1] > 0

df_raw0.head()

## 8. Standardize column names (`normalize_columns`)

> 💡 **Why this function?** Inconsistent column names (mixed case, spaces,
> dashes) are a silent source of bugs: `Net Profit`, `net_profit`, and
> `net-profit` are three different keys to pandas. Normalizing once, up
> front, keeps every downstream step predictable.

### Problem

Standardized column names reduce bugs and make pipelines consistent.

### Approach

Create a function called `normalize_columns` that receives a list of
column names as an argument and returns a list of normalized names.

The function should apply the following format:

-   [lowercase](https://docs.python.org/3/library/stdtypes.html#str.lower)
-   [remove extra
    spaces](https://docs.python.org/3/library/stdtypes.html#str.strip)
-   [dashes →
    underscores](https://docs.python.org/3/library/stdtypes.html#str.replace)
-   [keep only alphanumeric and
    underscore](https://docs.python.org/3/library/stdtypes.html#str.isalnum)
-   collapse repeated underscores

**Code Task 5.1.8.1**:

In [ ]:
def normalize_columns(columns: list[str]) -> list[str]:
    """
    Normalize column names to a consistent snake_case style.

    Parameters
    ----------
    columns
        Original column names.

    Returns
    -------
    list of str
        Normalized column names.
    """
    out: list[str] = []
    for c in columns:
        c = str(c)
        c = ...  # strip
        c = ... # lower
        c = ... # replace space by underscore
        c = ... # replace dash by underscore
        # check if ch is alpha numeric
        c = "".join(ch for ch in c if ... or ch == "_")
        while "__" in c:
            c = ...  # replace double underscore by single underscore
        out.append(c)
    return out

### Checkpoint

> 🧪 This checkpoint asserts the column names are **unchanged** after
> normalization — which, for these particular files, means they were
> already clean. (See the note right after it.)

**Code 5.1.8.2**:

In [ ]:
df_tmp0 = df_raw0.copy()
df_tmp0.columns = normalize_columns([str(c) for c in df_tmp0.columns])

assert df_tmp0.columns.equals(df_raw0.columns), (
    "Column names changed after normalization. "
    "This dataset may contain spaces/dashes/case differences that were "
    "standardized by `normalize_columns`."
)

> 📊 Depending on the file, you may or may not see changes after
> normalization. **Either outcome is fine** — we keep this step so the
> pipeline is robust to messy column names in *other* datasets, even if
> today's files happen to be tidy already.

## 9. Clean the dataset (`clean_financials`)

> 💡 **Why this function?** Models need **numbers**. Any column that
> arrived as text (or with stray placeholders) must become numeric, and
> anything that can't be parsed should become a missing value rather than
> crash the pipeline.

### Problem

We want numeric features for modeling.

Some columns may be strings (or contain placeholders). We will convert
all columns to numeric where possible. Non-numeric entries become
missing values.

### Approach

Create a function called `clean_financials` that receives a DataFrame as
an argument and returns a new version of the DataFrame with numeric
values and normalized column names.

Key points:

-   [copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html)
-   [to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)

> ⚠️ **Important note about labels.** Some files contain `bankrupt`, some
> do not. This function must work for **both** — it never assumes the
> target column exists.

**Code 5.1.9.1**:

In [ ]:
def clean_financials(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean dataset for modeling (simple version).

    Parameters
    ----------
    df
        Raw DataFrame extracted from JSON.

    Returns
    -------
    pandas.DataFrame
        Cleaned DataFrame with normalized columns and numeric values where
        possible.
    """
    df2 = df.copy()
    df2.columns = normalize_columns([str(c) for c in df2.columns])

    for col in df2.columns:
        df2[col] = pd.to_numeric(df2[col], errors="coerce")

    return df2

### Checkpoint

> 🧪 Confirms the cleaned frame has the **same shape** as the raw one — we
> changed *types*, not the number of rows or columns.

**Code 5.1.9.2**:

In [ ]:
df_clean0 = clean_financials(df_raw0)

assert isinstance(df_clean0, pd.DataFrame)
assert df_clean0.shape == df_raw0.shape
df_clean0.head()

## 10. Compose everything into `wrangle()`

> 🧱 **This is the payoff.** Each helper did one small job; `wrangle()`
> chains them into a single call. From here on — and across L2, L3, and
> L4 — turning a raw file into a clean table is just `wrangle(path)`.

### Problem

We want one function that does the entire transformation for any dataset
file.

### Approach

Create a method called `wrangle` that receives the path to a file as an
argument and returns the result of applying the previously created
methods: `load_json`, `extract_records`, `records_to_frame`, and
`clean_financials`.

Key points:

-   Load JSON payload
-   Extract record list (`data` or `observations`)
-   Convert records to DataFrame
-   Clean (normalize columns + numeric conversion)

> 📌 **Carry this forward:** in L2–L4 you'll write `from data import
> wrangle`. That `data.py` module is exactly this function (and its
> helpers), persisted by the platform outside the notebook.

**Code 5.1.10.1**:

In [ ]:
def wrangle(path: str | Path) -> pd.DataFrame:
    """
    Load + transform a bankruptcy dataset into a clean DataFrame.

    Parameters
    ----------
    path
        Path to a ``.json`` or ``.json.gz`` file.

    Returns
    -------
    pandas.DataFrame
        Cleaned DataFrame ready for analysis.
    """
    payload = load_json(path)
    records = extract_records(payload)
    df_raw = records_to_frame(records)
    return clean_financials(df_raw)

### Checkpoint

> 🧪 Runs the full pipeline end-to-end on one file and asserts the result
> is a non-empty DataFrame with **numeric columns** — proof that the
> composed `wrangle()` works.

**Code 5.1.10.2**:

In [ ]:
df_w0 = wrangle(paths[0])

assert isinstance(df_w0, pd.DataFrame)
assert df_w0.shape[0] > 0
assert df_w0.select_dtypes(include=["number"]).shape[1] > 0

## 11. Load all 4 datasets (Poland/Taiwan × with/without target)

### Problem

Load all 4 datasets (DataFrames) into a single dictionary.

### Approach

We will store every file path in a dictionary called `datasets_path` and
load each path using the `wrangle` function into a variable called
`frames`.

> 🔍 The four keys encode the full grid: `poland_features`,
> `poland_full`, `taiwan_features`, `taiwan_full`. The `*_features` files
> have **no** target; the `*_full` files **do**.

Key points:

-   Use [dictionary
    comprehension](https://docs.python.org/3/tutorial/datastructures.html#dictionaries)
    to create the `frames` variable.

**Code 5.1.11.1**:

In [ ]:
datasets_path: dict[str, Path] = {
    "poland_features": next(
        p for p in poland_paths if "features" in p.name.lower()
    ),
    "poland_full": next(
        p for p in poland_paths if "features" not in p.name.lower()
    ),
    "taiwan_features": next(
        p for p in taiwan_paths if "features" in p.name.lower()
    ),
    "taiwan_full": next(
        p for p in taiwan_paths if "features" not in p.name.lower()
    ),
}

datasets_path

**Code 5.1.11.2**:

In [ ]:
frames: dict[str, pd.DataFrame] = {
    name: wrangle(path) for name, path in datasets_path.items()
}

{k: v.shape for k, v in frames.items()}

### Checkpoint: identifiers exist

> 🧪 These asserts encode our expectations about the grid: Poland uses
> `company_id`, Taiwan uses `id`, and **only** the `*_full` frames carry
> `bankrupt`. If any expectation is wrong, we find out here.

**Code 5.1.11.3**:

In [ ]:
assert "company_id" in frames["poland_features"].columns
assert "company_id" in frames["poland_full"].columns
assert "id" in frames["taiwan_features"].columns
assert "id" in frames["taiwan_full"].columns

assert "bankrupt" not in frames["poland_features"].columns
assert "bankrupt" in frames["poland_full"].columns
assert "bankrupt" not in frames["taiwan_features"].columns
assert "bankrupt" in frames["taiwan_full"].columns

## 12. Validate expected columns (using the Data Dictionary)

This step is just to ensure that everything was done correctly. We
usually need to verify that everything was loaded as expected and that
all required columns are present. Otherwise, we need to check what went
wrong and why.

From the data dictionary:

-   Poland features: `feat_1` … `feat_64`
-   Taiwan features: `feat_1` … `feat_95`
-   The target `bankrupt` exists only in the "full" datasets.

> 🔍 **What to look for:** every expected `feat_*` column is present, the
> right identifier column exists, and `bankrupt` appears **only** where
> expected. A "missing features" message would point straight at a
> wrangling bug.

**Code 5.1.12.1**:

In [ ]:
def expected_feature_set(n_feats: int) -> set[str]:
    """Return expected feature columns: feat_1..feat_n."""
    return {f"feat_{i}" for i in range(1, n_feats + 1)}

poland_feats = expected_feature_set(64)
taiwan_feats = expected_feature_set(95)

checks = [
    ("poland_features", "company_id", poland_feats, False),
    ("poland_full", "company_id", poland_feats, True),
    ("taiwan_features", "id", taiwan_feats, False),
    ("taiwan_full", "id", taiwan_feats, True),
]

for name, id_col, feat_set, has_target in checks:
    df_ = frames[name]
    cols = set(df_.columns)

    assert id_col in cols, f"{name}: missing id column {id_col!r}"

    missing_feats = feat_set - cols
    assert not missing_feats, (
        f"{name}: missing features (sample): "
        f"{sorted(missing_feats)[:10]}"
    )

    if has_target:
        assert "bankrupt" in cols, f"{name}: expected 'bankrupt'"
    else:
        assert "bankrupt" not in cols, f"{name}: did not expect 'bankrupt'"

print("All column checks passed!")

> 📊 **`All column checks passed!`** means all four frames match the data
> dictionary exactly: correct identifiers, the full `feat_*` set (64 for
> Poland, 95 for Taiwan), and `bankrupt` present only in the `*_full`
> frames. The wrangling pipeline is validated.

## 13. Target checks + imbalance ratio (only for labeled datasets)

> 💡 **Why this matters now.** Bankruptcy is **rare**. A dataset where
> only a sliver of companies actually fail is called **imbalanced**, and
> imbalance quietly breaks naive models and naive metrics — a model that
> predicts "nobody goes bankrupt" can look 95%+ accurate while catching
> *zero* failures. Before we build any model (in L2), we measure how
> skewed the data is.

> 🧮 **The imbalance ratio**
>
> $$\text{imbalance ratio} = \frac{\text{count of majority class}}{\text{count of minority class}}$$
>
> -   A value near **1** → roughly balanced classes.
> -   A **large** value → severe imbalance (the minority class is rare).
>
> *Worked example:* 950 surviving firms and 50 bankrupt firms give a
> ratio of `950 / 50 = 19` — for every bankruptcy there are 19
> survivors.

> ⚠️ **Caveat / foreshadowing:** this single number is *why* L2 cannot
> rely on accuracy. The rarer the positive class, the more we will lean
> on precision, recall, and PR–AUC instead.

### Problem

Bankruptcy is usually rare. Class imbalance affects baselines and
evaluation.

### Approach

To assess the degree of class imbalance, we first ensure that the target
variable `bankrupt` is properly encoded as a binary outcome with values
`{0, 1}`. We then compute the imbalance ratio by counting the number of
observations in each class and dividing the size of the majority class
by the size of the minority class. This ratio provides a simple and
interpretable measure of how skewed the dataset is: values close to 1
indicate balanced classes, while larger values signal increasing
imbalance, which can strongly influence baseline models and evaluation
metrics.

-   Confirm `bankrupt` is binary `{0, 1}`
-   Compute imbalance ratio = majority_count / minority_count

Key points:

-   [astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)
-   [dropna](https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html)
-   [max](https://pandas.pydata.org/docs/reference/api/pandas.Series.max.html)
-   [min](https://pandas.pydata.org/docs/reference/api/pandas.Series.min.html)
-   [value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

**Code Task 5.1.13.1**:

In [ ]:
def imbalance_ratio(y: pd.Series) -> float:
    """Compute imbalance ratio: majority_count / minority_count."""
    s = ...  # drop Nan from `y` and cast values to int
    counts = ...  # counts of unique values from `s`

    if counts.shape[0] < 2:
        # If the size of `counts` is less than 2, the result is invalid,
        # as it should contain counts for both values (0 and 1).
        return float("nan")

    majority = ...  # get the max from the `counts`
    minority = ...  # get the min from the `counts`

    if minority == 0.0:
        # division by zero is invalid, so returns "inf"
        return float("inf")

    return ... # calculate the division between majority and minority

**Code 5.1.13.2**:

In [ ]:
labeled = {
    "poland_full": frames['poland_full'],  # get the "poland_full" from `frames`
    "taiwan_full": frames['taiwan_full'], # get the "taiwan_full" from `frames`
}

for name, df_ in labeled.items():
    vals = set(df_["bankrupt"].dropna().astype(int).unique())
    assert vals.issubset({0, 1}), f"{name} target not binary: {sorted(vals)}"

    ir = imbalance_ratio(df_["bankrupt"]) # get the imbalance ratio for "bankrupt" from `df_`
    print(f"{name}: imbalance ratio = {ir:.2f}")

> 📊 **Reading the imbalance ratios.** Each labeled dataset prints a
> ratio well above 1, confirming that surviving firms vastly outnumber
> bankrupt ones. Keep these numbers in mind: they are the reason the next
> notebook spends so much effort on metrics beyond accuracy.

# Wrap-up

You now have a clear, repeatable path from **compressed JSON** to a
**clean DataFrame**, and you can handle two realistic complications:

1.  Different datasets store records under different keys:

    -   Poland uses `data`
    -   Taiwan uses `observations`

2.  Some datasets include labels (`bankrupt`) and others are
    features-only.

> 🧠 **The one function to remember:** `wrangle(path)`. It composes
> `load_json` → `extract_records` → `records_to_frame` →
> `clean_financials`, and it is exactly what L2–L4 import via
> `from data import wrangle`.

➡️ **Next:** a clean table is only step one. In the next notebook we'll
build on these clean DataFrames to create train/validation/test splits
and establish **honest baselines** — and we'll discover why, on
imbalanced data, accuracy alone can badly mislead us.